# Import Dataset, drop 0 columns

In [18]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler


def clean_csv_files(folder_path):
  cleaned_dataframes = {}

  # List of columns to drop
  columns_to_drop = [
      "status", "start_date", "end_date", "window_start_date", "window_end_date",
      "emails", "devs", "emails_thread_starter", "emails_thread_starter_word_count",
      "emails_thread_starter_characters", "emails_threads", "emails_threads_word_count",
      "emails_threads_characters", "emails_no_replies", "emails_no_replies_word_count",
      "emails_no_replies_characters", "emails_jira", "most_complex_unit_loc",
      "most_complex_unit_mcabe_index", "total_number_of_files", "number_of_files_main",
      "lines_of_code_main", "number_of_files_test", "lines_of_code_test",
      "test_vs_main_lines_of_code_percentage", "number_of_files_generated",
      "lines_of_code_generated", "number_of_files_build_and_deployment",
      "lines_of_code_build_and_deployment", "negligible_risk_file_size_count",
      "low_risk_file_size_count", "medium_risk_file_size_count", "high_risk_file_size_count",
      "very_high_risk_file_size_count", "negligible_risk_file_size_loc", "low_risk_file_size_loc",
      "medium_risk_file_size_loc", "high_risk_file_size_loc", "very_high_risk_file_size_loc",
      "number_of_units", "lines_of_code_in_units", "lines_of_code_outside_units",
      "unit_size_negligible_risk_loc", "unit_size_negligible_risk_count", "unit_size_low_risk_loc",
      "unit_size_low_risk_count", "unit_size_medium_risk_loc", "unit_size_medium_risk_count",
      "unit_size_high_risk_loc", "unit_size_high_risk_count", "unit_size_very_high_risk_loc",
      "unit_size_very_high_risk_count", "conditional_complexity_negligible_risk_loc",
      "conditional_complexity_negligible_risk_count", "conditional_complexity_low_risk_loc",
      "conditional_complexity_low_risk_count", "conditional_complexity_medium_risk_loc",
      "conditional_complexity_medium_risk_count", "conditional_complexity_high_risk_loc",
      "conditional_complexity_high_risk_count", "conditional_complexity_very_high_risk_loc",
      "conditional_complexity_very_high_risk_count", "conditional_complexity_high_plus_risk_count",
      "conditional_complexity_high_plus_risk_loc", "number_of_contributors",
      "duplication_number_of_duplicates", "duplication_number_of_files_with_duplicates",
      "duplication_number_of_duplicated_lines", "duplication_percentage", "unit_duplicates_count", "releases"
  ]

  for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
      file_path = os.path.join(folder_path, filename)

      # Load CSV file
      df = pd.read_csv(file_path)

      # Drop specified columns
      df = df.drop(
          columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')

      key = os.path.splitext(filename)[0]
      cleaned_dataframes[key] = df

  return cleaned_dataframes

folder_path = "./scraper-output"
cleaned_data = clean_csv_files(folder_path)


# Clean data

In [19]:
import pandas as pd
import numpy as np

for key, df in cleaned_data.items():
    # Replace NaN values in numerical columns with 0
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = df[col].fillna(0)

    # Replace NaN and blank/empty values in 'programming_lang' column with the mode
    if 'programming_lang' in df.columns:
        # Calculate mode value
        mode_value = df['programming_lang'].mode()[0] if not df['programming_lang'].mode().empty else 'Unknown'
        
        # Replace NaN values with the mode
        df['programming_lang'] = df['programming_lang'].fillna(mode_value)
        
        # Replace blank or whitespace-only values with the mode
        df['programming_lang'] = df['programming_lang'].replace(r'^\s*$', mode_value, regex=True)

# Compute PCA to rank relevance of features

In [20]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def perform_pca_on_each(cleaned_data):
  feature_importance_list = []

  for key, df in cleaned_data.items():
    # print("key is ", key)
    # Store the dropped columns before transformation
    dropped_cols = df[['project', 'measurement_month', 'programming_lang']]
    # Exclude 'project', 'measurement_month', and 'programming_lang' columns
    features = df.drop(
        columns=['project', 'measurement_month', 'programming_lang'], errors='ignore')

    # Handle missing values - fill or drop NaNs
    features = features.fillna(0)

    # Drop columns with zero variance
    features = features.loc[:, features.var() > 0]

    # Check if there are any numeric features left
    numeric_features = features.select_dtypes(include=[np.number])
    if numeric_features.empty:
      print(
          f"Warning: No numeric features left for PCA in {key}. Skipping PCA.")
      continue

    # Standardize the data
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(numeric_features)
    scaled_df = pd.DataFrame(scaled_features, columns=numeric_features.columns)
    scaled_features_data[key] = pd.concat([dropped_cols.reset_index(drop=True), scaled_df], axis=1)
    # Perform PCA
    pca = PCA()
    pca.fit(scaled_features)
    # print("names ", numeric_features.columns)

    # Collect feature importance
    feature_importance = dict(
        zip(numeric_features.columns, pca.explained_variance_ratio_))
    # print("feature imp ", feature_importance)
    feature_importance_list.append(feature_importance)

  # Compute average importance across all DataFrames
  avg_feature_importance = {}
  for feature_dict in feature_importance_list:
    for feature, importance in feature_dict.items():
      if feature not in avg_feature_importance:
        avg_feature_importance[feature] = []
      avg_feature_importance[feature].append(importance)

  # Compute final average
  avg_feature_importance = {feature: sum(
      values) / len(values) for feature, values in avg_feature_importance.items()}
  # print("avg is ", avg_feature_importance)
  # Rank features by average importance
  ranked_features = sorted(avg_feature_importance.items(),
                           key=lambda x: x[1], reverse=True)

  # Display ranked features
  print("Final Ranked Features by Average Importance:")
  for feature, importance in ranked_features:
    print(f"{feature}: {importance:.4f}")

scaled_features_data = {}
# Perform PCA on each DataFrame and compute overall importance
perform_pca_on_each(cleaned_data)

# print("data is =  ", cleaned_data)

Final Ranked Features by Average Importance:
commits: 0.3743
authors: 0.1941
committers: 0.1070
minor_contributors: 0.0689
major_contributors: 0.0564
directories: 0.0438
top_level_dirs: 0.0367
active_days: 0.0298
files_modified: 0.0242
files_added: 0.0195
files_deleted: 0.0156
files_renamed: 0.0124
added_lines: 0.0094
deleted_lines: 0.0072
new_contributors: 0.0054
avg_files_modified_commit: 0.0040
code: 0.0028
blanks: 0.0020
files: 0.0014
comments: 0.0009
lines: 0.0005
stars: 0.0003
forks: 0.0002
open_prs: 0.0002
closed_prs: 0.0001
merged_prs: 0.0000
stale_prs: 0.0000
deploys: 0.0000


In [21]:
status_data = pd.read_csv("./project-status.csv")
status_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 154 entries, 0 to 153
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   project  154 non-null    object
 1   status   154 non-null    object
dtypes: object(2)
memory usage: 2.5+ KB


# Interaction analysis on most important features

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Filter out projects with fewer than 10 data points
scaled_features_data = {project: df for project,
                df in scaled_features_data.items() if len(df) >= 10}

# Filter status_data to only include projects present in cleaned_data
status_data_filtered = status_data[status_data['project'].isin(
    scaled_features_data.keys())]


# Merge project status into each dataframe
def merge_status(scaled_features_data, status_data):
  status_dict = status_data.set_index('project')['status'].to_dict()
  for project, df in scaled_features_data.items():
    df['status'] = status_dict.get(project, 'Unknown')
  return scaled_features_data

# Function to extract 1/10th segments and compute averages
def extract_average_feature(df, feature, num_bins=10):
  df = df.sort_values(by='measurement_month')  # Ensure time is sorted
  bin_size = max(1, len(df) // num_bins)  # Determine bin size
  averages = [df[feature].iloc[i *
                               bin_size: (i + 1) * bin_size].mean() for i in range(num_bins)]
  return averages

# Function to plot feature trends
def plot_feature(scaled_features_data, status_data, feature, num_bins=10):
  scaled_features_data = merge_status(scaled_features_data, status_data)

  # Collect averaged data for each status
  grouped_data = {status: [[] for _ in range(
      num_bins)] for status in status_data['status'].unique()}
  for project, df in scaled_features_data.items():
    if feature in df.columns:
      status = df['status'].iloc[0]
      averages = extract_average_feature(df, feature, num_bins)
      for i, avg in enumerate(averages):
        grouped_data[status][i].append(avg)
  # Compute overall average per bin for each status group
  vals[feature] = {}
  for status, bins in grouped_data.items():
    avg_series = [
        np.mean(bin_values) if bin_values else 0 for bin_values in bins]
    vals[feature][status] = avg_series

vals = {}
# Function to plot all features
def plot_all_features(scaled_features_data, status_data, features, num_bins=10):
  for feature in features:
    plot_feature(scaled_features_data, status_data, feature, num_bins)

plot_all_features(scaled_features_data, status_data, [
    'commits', 'authors', 'committers', 'minor_contributors', 'major_contributors',
    'directories', 'top_level_dirs', 'active_days', 'files_modified', 'files_added',
    'files_deleted', 'files_renamed', 'added_lines', 'deleted_lines', 'new_contributors',
    'avg_files_modified_commit'
])
cols = [
    'commits', 'authors', 'committers', 'minor_contributors', 'major_contributors',
    'directories', 'top_level_dirs', 'active_days', 'files_modified', 'files_added',
    'files_deleted', 'files_renamed', 'added_lines', 'deleted_lines', 'new_contributors',
    'avg_files_modified_commit']


In [23]:
print(vals)

{'commits': {'Graduated': [np.float64(0.33271645010827755), np.float64(0.2757820237175856), np.float64(0.16770282565828445), np.float64(0.0952721766138605), np.float64(0.037725394106032095), np.float64(0.023880527084703285), np.float64(0.025081834069805693), np.float64(-0.10320385662986048), np.float64(-0.2757211516925364), np.float64(-0.3470969114911576)], 'Retired': [np.float64(0.7400492523358307), np.float64(0.5464816612298061), np.float64(0.3327642354739959), np.float64(0.03554963927749066), np.float64(-0.028065146734648112), np.float64(-0.15024153757931663), np.float64(-0.20269275103622267), np.float64(-0.29749106639854833), np.float64(-0.3498382036257697), np.float64(-0.3670548244975532)]}, 'authors': {'Graduated': [np.float64(-0.21030585713942035), np.float64(-0.031926401134140775), np.float64(0.11353621526997415), np.float64(0.0929121396171306), np.float64(0.17935984875686126), np.float64(0.1742283583169537), np.float64(0.18300725799723702), np.float64(0.030116412936895556), np

In [24]:
import ipywidgets as widgets

# Dropdown widgets
y_axis_1 = widgets.Dropdown(options=cols, value='authors', description='Y-axis:1')
y_axis_2 = widgets.Dropdown(options=cols, value='commits', description='Y-axis:2')
# Function to update the plot
def update_plot(y_axis_1_, y_axis_2_):
    
    fig, ax1 = plt.subplots(figsize=(8, 6))
    ax1.plot(range(10), vals[y_axis_1_]['Graduated'], color="red", marker='o', label=y_axis_1_+'-Graduated')
    ax1.plot(range(10), vals[y_axis_1_]['Retired'], color="blue", marker='o', label=y_axis_1_+'-Retired')
    ax1.set_xlabel('Normalized Time (Bins)')
    ax1.set_ylabel(y_axis_1_, color='black')
    ax1.tick_params(axis='y', labelcolor='black')
    
    ax2 = ax1.twinx()
    ax2.plot(range(10), vals[y_axis_2_]['Graduated'], color="green", marker='o', label=y_axis_2_+'-Graduated')
    ax2.plot(range(10), vals[y_axis_2_]['Retired'], color="purple", marker='o', label=y_axis_2_+'-Retired')
    ax2.tick_params(axis='y', labelcolor='black')    
    ax2.set_ylabel(y_axis_2_, color='black')
    fig.legend()
    plt.show()

# Interactive widget
ui = widgets.VBox([y_axis_1, y_axis_2])
out = widgets.interactive_output(update_plot, {'y_axis_1_': y_axis_1, 'y_axis_2_': y_axis_2})

# Display widgets and plot
display(ui, out)

Output()

In [25]:
# Dropdown widgets
import pandas as pd
y_axis_1 = widgets.Dropdown(options=cols, value='authors', description='Y-axis:1')
y_axis_2 = widgets.Dropdown(options=cols, value='commits', description='Y-axis:2')
# Function to update the plot
def update_plot(y_axis_1_, y_axis_2_):
    df = pd.DataFrame({
        y_axis_1_+'-Graduated': vals[y_axis_1_]['Graduated'],
        y_axis_1_+'-Retired': vals[y_axis_1_]['Retired'],
        y_axis_2_+'-Graduated': vals[y_axis_2_]['Graduated'],
        y_axis_2_+'-Retired': vals[y_axis_2_]['Retired'],
    })
    corr = df.corr()
    fig, ax = plt.subplots(figsize=(6, 5))
    cax = ax.imshow(corr, cmap='coolwarm', interpolation='nearest')
    
    # Add colorbar
    fig.colorbar(cax)
    
    # Set axis labels
    ax.set_xticks(np.arange(len(corr.columns)))
    ax.set_yticks(np.arange(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45)
    ax.set_yticklabels(corr.columns)
    
    # Display values in cells
    for i in range(len(corr.columns)):
        for j in range(len(corr.columns)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha='center', va='center', color='black')
    
    plt.title("Correlation Heatmap")
    plt.show()

# Interactive widget
ui = widgets.VBox([y_axis_1, y_axis_2])
out = widgets.interactive_output(update_plot, {'y_axis_1_': y_axis_1, 'y_axis_2_': y_axis_2})

# Display widgets and plot
display(ui, out)


Output()

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from IPython.display import display

# Dropdown widgets
y_axis_1 = widgets.Dropdown(options=cols, value='authors', description='Y-axis:1')
y_axis_2 = widgets.Dropdown(options=cols, value='commits', description='Y-axis:2')

# Function to update the plot with correlation and p-values
def update_plot(y_axis_1_, y_axis_2_):
    df = pd.DataFrame({
        y_axis_1_+'-Graduated': vals[y_axis_1_]['Graduated'],
        y_axis_1_+'-Retired': vals[y_axis_1_]['Retired'],
        y_axis_2_+'-Graduated': vals[y_axis_2_]['Graduated'],
        y_axis_2_+'-Retired': vals[y_axis_2_]['Retired'],
    })
    
    corr_matrix = df.corr()
    p_values = pd.DataFrame(np.zeros((df.shape[1], df.shape[1])), columns=df.columns, index=df.columns)
    
    # Compute p-values
    for col1 in df.columns:
        for col2 in df.columns:
            _, p = stats.pearsonr(df[col1], df[col2])
            p_values.loc[col1, col2] = p
    
    fig, ax = plt.subplots(figsize=(6, 5))
    cax = ax.imshow(corr_matrix, cmap='coolwarm', interpolation='nearest')

    # Add colorbar
    fig.colorbar(cax)

    # Set axis labels
    ax.set_xticks(np.arange(len(corr_matrix.columns)))
    ax.set_yticks(np.arange(len(corr_matrix.columns)))
    ax.set_xticklabels(corr_matrix.columns, rotation=45)
    ax.set_yticklabels(corr_matrix.columns)

    # Display correlation and p-values in cells
    for i in range(len(corr_matrix.columns)):
        for j in range(len(corr_matrix.columns)):
            corr_val = corr_matrix.iloc[i, j]
            p_val = p_values.iloc[i, j]
            text = f"{corr_val:.2f}\n(p={p_val:.3f})"
            ax.text(j, i, text, ha='center', va='center', color='black')

    plt.title("Correlation Heatmap with P-Values")
    plt.show()

# Interactive widget
ui = widgets.VBox([y_axis_1, y_axis_2])
out = widgets.interactive_output(update_plot, {'y_axis_1_': y_axis_1, 'y_axis_2_': y_axis_2})

# Display widgets and plot
display(ui, out)



Output()

In [28]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

# Get min/max time from dataset


# Dropdown widgets
y_axis = widgets.Dropdown(options=cols, value=cols[0], description='Y-axis:')
x_axis = widgets.Dropdown(options=cols, value=cols[1], description='X-axis:')

# Function to update scatter plot with time encoding
def update_plot(x_col, y_col):
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Extract numerical values
    x_graduated = vals[x_col]['Graduated']
    y_graduated = vals[y_col]['Graduated']
    time = [i for i in range(10)]
    
    
    # Normalize time for color mapping
    norm = plt.Normalize(0, 10)
    cmap = plt.get_cmap('viridis')
    
    # Scatter plot with color-mapped time encoding
    sc1 = ax.scatter(x_graduated, y_graduated, c=time, cmap=cmap, alpha=0.7, label='Graduated', norm=norm)
    
    # Colorbar to show time progression
    cbar = plt.colorbar(sc1, ax=ax)
    cbar.set_label("Time")

    # Labels and legend
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.legend()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 6))
    # Extract numerical values
    time = [i for i in range(10)]
    
    x_retired = vals[x_col]['Retired']
    y_retired = vals[y_col]['Retired']
    
    
    # Normalize time for color mapping
    norm = plt.Normalize(0, 10)
    cmap = plt.get_cmap('viridis')
    
    # Scatter plot with color-mapped time encoding
    sc2 = ax.scatter(x_retired, y_retired, c=time, cmap=cmap, alpha=0.7, label='Retired', norm=norm)
    
    # Colorbar to show time progression
    cbar = plt.colorbar(sc1, ax=ax)
    cbar.set_label("Time")

    # Labels and legend
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.legend()
    plt.show()

# Interactive widget setup
ui = widgets.VBox([x_axis, y_axis])
out = widgets.interactive_output(update_plot, {'x_col': x_axis, 'y_col': y_axis})

# Display widgets and plot
display(ui, out)



Output()